In [1]:
import pandas as pd
import requests
import io
from zipfile import ZipFile
from typing import List, Dict, Any

In [2]:
def extract(url: str) -> pd.DataFrame:
    """Fetches a zip file from a URL and reads the CSV into a DataFrame."""
    response: requests.Response = requests.get(url)
    zip_file: ZipFile = ZipFile(io.BytesIO(response.content))

    fname: str = zip_file.namelist()[0]

    with zip_file.open(fname) as f:
        df: pd.DataFrame = pd.read_csv(f)

    df.columns = df.columns.str.strip()
    return df

daily_url: str = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref.zip"
hist_url: str = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist.zip"

daily: pd.DataFrame = extract(daily_url)
historical: pd.DataFrame = extract(hist_url)

In [3]:
currencies: List[str] = ['USD', 'SEK', 'GBP', 'JPY']

# Get the latest daily rates
daily_latest: pd.Series = daily.iloc[0]

daily_rates: Dict[str, float] = {}
hist_means: Dict[str, float] = {}
results: List[Dict[str, Any]] = []

for currency in currencies:
    numeric_historical = pd.to_numeric(historical[currency], errors='coerce')

    results.append({
        'Currency Code': currency,
        'Rate': float(daily_latest[currency]),
        'Mean Historical Rate': round(float(numeric_historical.mean()), 4)
    })
results: pd.DataFrame = pd.DataFrame(results)

In [4]:
markdown_table: str = results.to_markdown(index=False)
fname: str = "exchange_rates.md"

with open(fname, "w", encoding="utf-8") as f:
    f.write(markdown_table)